# BSH Product-Level Model: Enzyme + Amine + Bile Acid Core

**Redesigned model** that preserves hydroxylation-pattern specificity instead of collapsing to pair-level.

Each sample is an **(Enzyme, Amine, Hydroxylation pattern)** triple.

**Three feature blocks:**
- Enzyme (1024-dim): ProtT5 per-residue, conservation-guided (noncons_max vs full_protein)
- Amine (41/768/91-dim): physchem_onehot vs MolT5-base vs hybrid
- Bile acid core (12-dim): Positional encoding (C3/C7/C12 status) + count features

**Label schemes:**
- `active_approach2` (existing median-based)
- `active_majority` (2/3 replicates must detect) — NEW

**Experiment grid:** 2 enzyme × 3 amine × 2 label schemes × 10 seeds = 120 experiments

## Section 1: Imports & Config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import h5py
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, log_loss
)

import xgboost as xgb

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

from Bio import SeqIO

# --- Paths ---
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PRODUCT_DIR = OUTPUT_DIR / "model_outputs" / "product_level_model"
PRODUCT_DIR.mkdir(exist_ok=True, parents=True)

# --- Constants ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

N_SPLITS = 10
SEEDS = [42, 123, 456, 789, 1011, 2022, 3033, 4044, 5055, 6066]

# Amines to exclude (no SMILES available + control)
EXCLUDE_AMINES = {'serotonin', 'tyramine', 'glyglycine', 'cystine', 'unconjugated'}

# XGBoost config
XGB_PARAMS = dict(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    reg_alpha=1.0, reg_lambda=5.0,
    subsample=0.7, colsample_bytree=0.7,
    min_child_weight=5,
    random_state=42, early_stopping_rounds=30,
    eval_metric='logloss', n_jobs=-1
)

# Physicochemical descriptor count
N_PHYSCHEM = 15
PHYSCHEM_NAMES = [
    'MolWt', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'AromaticRings',
    'AliphaticRings', 'FractionCSP3', 'HeavyAtoms', 'AmideBonds',
    'ValenceElectrons', 'MaxPartialCharge', 'MinPartialCharge', 'BalabanJ'
]

# Bile acid encoding
PATTERN_INFO = {
    '3a7k':    ('a', 'k', None, 1, 1),
    '3k7a':    ('k', 'a', None, 1, 1),
    '3a12k':   ('a', None, 'k', 1, 1),
    '3k12a':   ('k', None, 'a', 1, 1),
    '3a7a12k': ('a', 'a', 'k',  2, 1),
    'Mono':    (None, None, None, 1, 0),
    'Di':      (None, None, None, 2, 0),
    'Tri':     (None, None, None, 3, 0),
}
BA_FEATURE_NAMES = [
    'C3_aOH', 'C3_keto', 'C3_unspec',
    'C7_aOH', 'C7_keto', 'C7_unspec',
    'C12_aOH', 'C12_keto', 'C12_unspec',
    'n_OH', 'n_keto', 'is_specific',
]
BA_DIM = len(BA_FEATURE_NAMES)  # 12

print(f"Output directory: {PRODUCT_DIR}")
print("Setup complete.")

## Section 2: Load All Data

In [ ]:
%%time
# --- Per-residue ProtT5 embeddings ---
h5_path = DATA_DIR / "Seqs_list_total_per_residue.h5"
per_residue_embeddings = {}
with h5py.File(h5_path, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        per_residue_embeddings[uniprot_id] = f[key][:]
print(f"Per-residue embeddings: {len(per_residue_embeddings)} enzymes")

# --- Full protein embeddings (mean-pooled, for baseline) ---
h5_full = DATA_DIR / "Seqs_list_total.h5"
full_embeddings = {}
with h5py.File(h5_full, 'r') as f:
    for key in f.keys():
        uniprot_id = key.split('_')[-1]
        full_embeddings[uniprot_id] = f[key][:]
print(f"Full protein embeddings: {len(full_embeddings)} enzymes")

# --- Conservation scores ---
df_cons = pd.read_csv(OUTPUT_DIR / "conservation_scores.csv")
core_mask = df_cons['gap_fraction'] < 0.5
df_core = df_cons[core_mask].copy()
print(f"Conservation: {len(df_core)} core positions (gap < 0.5)")

# --- MSA alignment mapping ---
alignment_to_seq = {}
for record in SeqIO.parse(OUTPUT_DIR / "bsh_aligned.fasta", 'fasta'):
    parts = record.id.split('_')
    uniprot_id = parts[-1] if len(parts) > 1 else record.id
    seq = str(record.seq)
    mapping = {}
    seq_pos = 0
    for aln_pos, char in enumerate(seq):
        if char != '-':
            mapping[aln_pos] = seq_pos
            seq_pos += 1
    alignment_to_seq[uniprot_id] = mapping
overlap = set(alignment_to_seq.keys()) & set(per_residue_embeddings.keys())
print(f"Enzymes with alignment + per-residue embeddings: {len(overlap)}")

# --- Activity data (PRODUCT-LEVEL, no aggregation) ---
df_activity = pd.read_csv(OUTPUT_DIR / "enzyme_amine_activity.csv")
controls = ['CTRL1', 'CTRL2', 'CTRL3', 'CTRL4', 'CTRL5', 'CTRL6', 'CTRL7']
canonical = ['taurine', 'glycine']
df_activity = df_activity[~df_activity['Enzyme'].isin(controls)]
df_activity = df_activity[~df_activity['Amine'].isin(canonical)]
df_activity = df_activity[~df_activity['Amine'].isin(EXCLUDE_AMINES)]
# Merge hydroxyl variants
df_activity['Hydroxyl'] = df_activity['Hydroxyl'].replace('3a,7a,12k', '3a7a12k')
print(f"\nProduct-level rows (after filtering): {len(df_activity)}")
print(f"  Enzymes: {df_activity['Enzyme'].nunique()}, Amines: {df_activity['Amine'].nunique()}")
print(f"  Hydroxylation patterns: {df_activity['Hydroxyl'].nunique()}")
print(f"  Active (approach2): {df_activity['active_approach2'].sum()} ({df_activity['active_approach2'].mean():.1%})")
print(f"\nHydroxylation pattern distribution:")
print(df_activity['Hydroxyl'].value_counts().sort_index())

# --- Amine SMILES ---
df_smiles = pd.read_excel(DATA_DIR / "bsh_reactants_SMILES_corrected.xlsx")

name_map = {
    '2,3-Diaminopropinoic Acid': '2,3_diaminopropionic acid',
    '2-aminophenol': '2_aminophenol',
    '3-methoxytyramine HCl': '3_methoxytyramine',
    '4-aminophenol': '4_aminophenol',
    'L-Alanine': 'alanine',
    'L-Arginine': 'arginine',
    'Asparagine': 'asparagine',
    'Cadaverine': 'cadaverine',
    'L-Citrulline': 'citrulline',
    'L-Cysteine': 'cysteine',
    'Dopamine HCl': 'dopamine',
    'gamma-Aminobutyric acid >99%': 'gaba',
    'L-Glutamine': 'glutamine',
    'Glycyl-L-Valine': 'glyglycine',
    'L-Histidine': 'histidine',
    'L-Lysine': 'lysine',
    'L-Methionine': 'methionine',
    'L-Ornithine monohydrochloride': 'ornithine',
    'L-Phenylalanine': 'phenylalanine',
    'DL-Proline': 'proline',
    'Putrescine': 'putrescine',
    'L-Serine': 'serine',
    'L-Threonine': 'threonine',
    'Tryptamine': 'tryptamine',
}

amine_mols = {}
for _, row in df_smiles.iterrows():
    name = row['Compound_Name']
    smiles = row['SMILES']
    norm_name = name_map.get(name, name.lower().replace(' ', '_').replace('-', '_'))
    if pd.isna(smiles):
        continue
    smiles_clean = smiles.split('.')[0]
    mol = Chem.MolFromSmiles(smiles_clean)
    if mol is not None:
        amine_mols[norm_name] = mol

amines_needed = df_activity['Amine'].unique()
print(f"\nAmines needed: {len(amines_needed)}, parsed from SMILES: {len(amine_mols)}")
missing_amines = set(amines_needed) - set(amine_mols.keys())
if missing_amines:
    print(f"  Missing from SMILES: {missing_amines}")

# --- MolT5-base embeddings ---
molt5_base_path = DATA_DIR / "molt5_base_amine_embeddings.csv"
repr_molt5_base = {}
if molt5_base_path.exists():
    df_m5b = pd.read_csv(molt5_base_path)
    for _, row in df_m5b.iterrows():
        name = row['amine']
        vec = row.drop('amine').values.astype(np.float32)
        repr_molt5_base[name] = vec
    print(f"\nMolT5-base: {len(repr_molt5_base)} amines, {len(vec)} dims")
else:
    print(f"\nWARNING: {molt5_base_path} not found!")

# --- Heatmap CSVs (for replicate labels) ---
df_heatmap_amines = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_amines_for_heatmap_manual.csv")
df_heatmap_subs = pd.read_csv(DATA_DIR / "NEW_Stage2_BAs_subs_for_heatmap_manual.csv")
print(f"\nHeatmap amines: {df_heatmap_amines.shape}")
print(f"Heatmap subs: {df_heatmap_subs.shape}")

## Section 3: Replicate-Aware Activity Labels

In [ ]:
# --- Parse product column names ---
def parse_product_col(col):
    """Parse product column name into (hydroxyl, amine, product_id)."""
    parts = col.rsplit('_', 1)
    if len(parts) != 2 or not parts[1].isdigit():
        return None, None, None
    product_id = parts[1]
    remainder = parts[0]
    known_hydroxyls = ['3a,7a,12k', '3a7a12k', '3a12k', '3a7k', '3k12a', '3k7a', 'Di', 'Mono', 'Tri']
    for h in sorted(known_hydroxyls, key=len, reverse=True):
        if remainder.startswith(h + '_'):
            amine = remainder[len(h)+1:]
            return h, amine, product_id
    return None, None, None

# --- Merge both heatmap files ---
df_raw = pd.merge(df_heatmap_subs, df_heatmap_amines, on=['filename', 'Code', 'Replicate'], how='outer')
df_raw = df_raw[df_raw['Code'].notna() & (df_raw['Code'] != 'NA')].copy()
print(f"Merged heatmap rows (excluding blanks): {len(df_raw)}")

# --- Parse all product columns ---
meta_cols = ['filename', 'Code', 'Replicate']
product_cols = [c for c in df_raw.columns if c not in meta_cols]

product_info = {}
for col in product_cols:
    h, a, pid = parse_product_col(col)
    if h is not None:
        # Normalize hydroxyl
        h_norm = h.replace('3a,7a,12k', '3a7a12k')
        product_info[col] = {'hydroxyl': h_norm, 'amine': a, 'product_id': pid}

print(f"Parsed product columns: {len(product_info)} / {len(product_cols)} total")

# --- Melt to long format ---
df_long = df_raw.melt(
    id_vars=['Code', 'Replicate'],
    value_vars=list(product_info.keys()),
    var_name='product',
    value_name='intensity'
)
df_long['amine'] = df_long['product'].map(lambda x: product_info[x]['amine'])
df_long['hydroxyl'] = df_long['product'].map(lambda x: product_info[x]['hydroxyl'])
df_long['intensity'] = df_long['intensity'].fillna(0)
df_long['detected'] = (df_long['intensity'] > 0).astype(int)

print(f"Long format rows: {len(df_long)}")
print(f"Unique enzymes: {df_long['Code'].nunique()}, amines: {df_long['amine'].nunique()}, patterns: {df_long['hydroxyl'].nunique()}")

In [ ]:
# --- Compute replicate stats per (Enzyme, Amine, Hydroxyl) ---
# Group by enzyme × amine × hydroxyl (summing across product IDs for same triple)
# A triple is "detected" in a replicate if ANY product with that (amine, hydroxyl) is detected
replicate_detection = df_long.groupby(['Code', 'amine', 'hydroxyl', 'Replicate']).agg(
    any_detected=('detected', 'max')  # 1 if any product detected in this replicate
).reset_index()

# Now count how many replicates detected each (enzyme, amine, hydroxyl)
replicate_counts = replicate_detection.groupby(['Code', 'amine', 'hydroxyl']).agg(
    n_reps=('any_detected', 'count'),
    n_detected=('any_detected', 'sum'),
).reset_index()

# Majority vote label: detected in >= 2 of 3 replicates
replicate_counts['active_majority'] = (replicate_counts['n_detected'] >= 2).astype(int)

print(f"Replicate count entries: {len(replicate_counts)}")
print(f"\nMajority vote label distribution:")
print(replicate_counts['active_majority'].value_counts())
print(f"Active rate: {replicate_counts['active_majority'].mean():.1%}")

# Detection consistency
print(f"\nDetection consistency:")
for n in sorted(replicate_counts['n_detected'].unique()):
    count = (replicate_counts['n_detected'] == n).sum()
    print(f"  {n}/{replicate_counts['n_reps'].mode().values[0]} replicates: {count} ({count/len(replicate_counts):.1%})")

In [ ]:
# --- Merge majority vote labels into activity dataframe ---
# Map enzyme codes: activity uses UniProt IDs, heatmap uses Code
# They should match — let's check
activity_enzymes = set(df_activity['Enzyme'].unique())
heatmap_enzymes = set(replicate_counts['Code'].unique())
enzyme_overlap = activity_enzymes & heatmap_enzymes
print(f"Activity enzymes: {len(activity_enzymes)}, Heatmap enzymes: {len(heatmap_enzymes)}")
print(f"Overlap: {len(enzyme_overlap)}")

# Merge majority label onto df_activity
df_activity = df_activity.merge(
    replicate_counts[['Code', 'amine', 'hydroxyl', 'n_detected', 'active_majority']],
    left_on=['Enzyme', 'Amine', 'Hydroxyl'],
    right_on=['Code', 'amine', 'hydroxyl'],
    how='left'
)

# Fill NaN majority labels with approach2 as fallback
n_matched = df_activity['active_majority'].notna().sum()
n_missing = df_activity['active_majority'].isna().sum()
print(f"\nMatched majority labels: {n_matched} ({n_matched/len(df_activity):.1%})")
print(f"Missing (fallback to approach2): {n_missing} ({n_missing/len(df_activity):.1%})")

df_activity['active_majority'] = df_activity['active_majority'].fillna(
    df_activity['active_approach2'].astype(int)
).astype(int)

# Compare label schemes
label_diff = (df_activity['active_majority'] != df_activity['active_approach2'].astype(int)).sum()
print(f"\nLabel disagreements: {label_diff} ({label_diff/len(df_activity):.1%})")
print(f"\nLabel comparison:")
print(pd.crosstab(df_activity['active_approach2'], df_activity['active_majority'],
                  rownames=['approach2'], colnames=['majority']))

# Save label comparison
label_comp = df_activity[['Enzyme', 'Amine', 'Hydroxyl', 'active_approach2', 'active_majority', 'n_detected']].copy()
label_comp.to_csv(PRODUCT_DIR / 'replicate_label_comparison.csv', index=False)
print(f"\nSaved: {PRODUCT_DIR / 'replicate_label_comparison.csv'}")

## Section 4: Build Feature Matrices

In [ ]:
# --- Helper functions ---

def get_nonconserved_embedding(enzyme_id, conservation_threshold=0.6, pooling='max'):
    """Extract and pool per-residue embeddings at non-conserved positions."""
    if enzyme_id not in per_residue_embeddings or enzyme_id not in alignment_to_seq:
        return None
    embed = per_residue_embeddings[enzyme_id]
    aln_map = alignment_to_seq[enzyme_id]
    variable_aln_positions = df_core[
        df_core['conservation_score'] < conservation_threshold
    ]['alignment_position'].values
    seq_positions = []
    for aln_pos in variable_aln_positions:
        if aln_pos in aln_map:
            seq_pos = aln_map[aln_pos]
            if seq_pos < len(embed):
                seq_positions.append(seq_pos)
    if len(seq_positions) == 0:
        return None
    selected = embed[seq_positions]
    if pooling == 'mean':
        return selected.mean(axis=0)
    elif pooling == 'max':
        return selected.max(axis=0)
    return selected.mean(axis=0)


def compute_physicochemical(mol):
    """Compute 15 RDKit physicochemical descriptors."""
    return np.array([
        Descriptors.MolWt(mol), Descriptors.MolLogP(mol), Descriptors.TPSA(mol),
        Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
        Descriptors.NumRotatableBonds(mol), Descriptors.NumAromaticRings(mol),
        Descriptors.NumAliphaticRings(mol), Descriptors.FractionCSP3(mol),
        Descriptors.HeavyAtomCount(mol), rdMolDescriptors.CalcNumAmideBonds(mol),
        Descriptors.NumValenceElectrons(mol), Descriptors.MaxPartialCharge(mol),
        Descriptors.MinPartialCharge(mol),
        Descriptors.BalabanJ(mol) if Descriptors.BalabanJ(mol) != 0 else 0.0,
    ], dtype=np.float32)


def encode_bile_acid(hydroxyl_value):
    """Positional encoding: C3/C7/C12 status + counts = 12 dims."""
    vec = np.zeros(BA_DIM, dtype=np.float32)
    info = PATTERN_INFO.get(hydroxyl_value)
    if info is None:
        return vec
    c3, c7, c12, n_oh, n_keto = info
    for pos_idx, status in enumerate([c3, c7, c12]):
        base = pos_idx * 3
        if status == 'a':
            vec[base] = 1.0
        elif status == 'k':
            vec[base + 1] = 1.0
        else:
            vec[base + 2] = 1.0
    vec[9] = n_oh
    vec[10] = n_keto
    vec[11] = 1.0 if hydroxyl_value not in ('Mono', 'Di', 'Tri') else 0.0
    return vec


def enzyme_holdout_split_seed(X, y, enzymes, seed, test_size=0.2, val_size=0.2):
    """Enzyme hold-out split with stratified activity profiles."""
    enzymes_arr = np.array(enzymes)
    unique_enzymes = np.unique(enzymes_arr)
    profiles = np.array([y[enzymes_arr == e].mean() for e in unique_enzymes])
    bins = pd.cut(profiles, bins=5, labels=False)
    train_val_enz, test_enz = train_test_split(
        unique_enzymes, test_size=test_size, random_state=seed, stratify=bins)
    tv_profiles = np.array([y[enzymes_arr == e].mean() for e in train_val_enz])
    tv_bins = pd.cut(tv_profiles, bins=5, labels=False)
    train_enz, val_enz = train_test_split(
        train_val_enz, test_size=val_size, random_state=seed, stratify=tv_bins)
    train_mask = np.isin(enzymes_arr, train_enz)
    val_mask = np.isin(enzymes_arr, val_enz)
    test_mask = np.isin(enzymes_arr, test_enz)
    return {
        'X_train': X[train_mask], 'y_train': y[train_mask],
        'X_val': X[val_mask], 'y_val': y[val_mask],
        'X_test': X[test_mask], 'y_test': y[test_mask],
        'train_enz': train_enz, 'val_enz': val_enz, 'test_enz': test_enz,
        'test_enzymes_arr': enzymes_arr[test_mask],
    }


def train_xgb_product(split_data):
    """Train regularized XGBoost, return metrics + model + feature importances."""
    X_train, y_train = split_data['X_train'], split_data['y_train']
    X_val, y_val = split_data['X_val'], split_data['y_val']
    X_test, y_test = split_data['X_test'], split_data['y_test']
    
    n_neg = (y_train == 0).sum()
    n_pos = max((y_train == 1).sum(), 1)
    
    model = xgb.XGBClassifier(
        scale_pos_weight=n_neg / n_pos,
        **XGB_PARAMS
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False
    )
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_proba_train = model.predict_proba(X_train)[:, 1]
    y_proba_val = model.predict_proba(X_val)[:, 1]
    
    metrics = {
        'train_logloss': log_loss(y_train, y_proba_train),
        'val_logloss': log_loss(y_val, y_proba_val),
        'test_logloss': log_loss(y_test, y_proba),
        'logloss_gap': log_loss(y_val, y_proba_val) - log_loss(y_train, y_proba_train),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_test, y_pred),
        'train_acc': model.score(X_train, y_train),
        'val_acc': model.score(X_val, y_val),
        'n_train': len(y_train), 'n_val': len(y_val), 'n_test': len(y_test),
    }
    
    return metrics, model

print("Helper functions defined.")

In [ ]:
%%time
# --- Build enzyme representations ---
enzyme_repr = {}

# noncons_max: conservation < 0.6, max-pooled (our optimized representation)
d = {}
for eid in overlap:
    emb = get_nonconserved_embedding(eid, conservation_threshold=0.6, pooling='max')
    if emb is not None:
        d[eid] = emb
enzyme_repr['noncons_max'] = d

# full_protein: whole-protein ProtT5 embedding (baseline reference)
enzyme_repr['full_protein'] = {eid: emb for eid, emb in full_embeddings.items()}

print(f"{'Enzyme repr':<20} {'Enzymes':>8} {'Dims':>6}")
print("-" * 36)
for name, d in enzyme_repr.items():
    sample = list(d.values())[0]
    print(f"{name:<20} {len(d):>8} {sample.shape[0]:>6}")

In [ ]:
# --- Build amine representations ---

# A: physchem_onehot (15 physicochemical + N one-hot)
all_amines_sorted = sorted(amines_needed)
amine_to_idx = {a: i for i, a in enumerate(all_amines_sorted)}
n_amines_onehot = len(all_amines_sorted)

repr_physchem_onehot = {}
for name in amines_needed:
    phys = compute_physicochemical(amine_mols[name]) if name in amine_mols else np.zeros(N_PHYSCHEM, dtype=np.float32)
    onehot = np.zeros(n_amines_onehot, dtype=np.float32)
    if name in amine_to_idx:
        onehot[amine_to_idx[name]] = 1.0
    repr_physchem_onehot[name] = np.concatenate([phys, onehot])

physchem_onehot_dim = N_PHYSCHEM + n_amines_onehot
print(f"A: physchem_onehot — {len(repr_physchem_onehot)} amines, {physchem_onehot_dim} dims")

# B: MolT5-base (768-dim)
molt5_base_dim = len(list(repr_molt5_base.values())[0]) if repr_molt5_base else 0
print(f"B: MolT5-base — {len(repr_molt5_base)} amines, {molt5_base_dim} dims")

# C: physchem_onehot + PCA(MolT5, n=50) = hybrid
PCA_DIMS = 50
repr_hybrid = {}
if repr_molt5_base:
    # PCA on MolT5 embeddings for amines we need
    molt5_amines_available = [a for a in amines_needed if a in repr_molt5_base]
    molt5_matrix = np.array([repr_molt5_base[a] for a in molt5_amines_available])
    pca = PCA(n_components=min(PCA_DIMS, molt5_matrix.shape[0], molt5_matrix.shape[1]))
    molt5_pca = pca.fit_transform(molt5_matrix)
    molt5_pca_dict = {a: molt5_pca[i] for i, a in enumerate(molt5_amines_available)}
    actual_pca_dims = molt5_pca.shape[1]
    
    for name in amines_needed:
        physchem_oh = repr_physchem_onehot[name]
        if name in molt5_pca_dict:
            pca_vec = molt5_pca_dict[name].astype(np.float32)
        else:
            pca_vec = np.zeros(actual_pca_dims, dtype=np.float32)
        repr_hybrid[name] = np.concatenate([physchem_oh, pca_vec])
    
    hybrid_dim = physchem_onehot_dim + actual_pca_dims
    print(f"C: hybrid (physchem_onehot + PCA MolT5) — {len(repr_hybrid)} amines, {hybrid_dim} dims")
    print(f"   PCA variance explained: {pca.explained_variance_ratio_.sum():.1%} ({actual_pca_dims} components)")
else:
    print("C: hybrid — SKIPPED (no MolT5 embeddings)")

# Collect amine representations
amine_reprs = {}
amine_reprs['physchem_onehot'] = (repr_physchem_onehot, physchem_onehot_dim)
if repr_molt5_base:
    amine_reprs['molt5_base'] = (repr_molt5_base, molt5_base_dim)
if repr_hybrid:
    amine_reprs['hybrid'] = (repr_hybrid, hybrid_dim)

print(f"\nAmine representations: {list(amine_reprs.keys())}")

In [ ]:
# --- Encode bile acid patterns ---
bile_acid_encodings = {}
for h in df_activity['Hydroxyl'].unique():
    enc = encode_bile_acid(h)
    bile_acid_encodings[h] = enc
    info = PATTERN_INFO.get(h, (None, None, None, 0, 0))
    print(f"  {h:12s} -> C3={str(info[0]):>4s} C7={str(info[1]):>4s} C12={str(info[2]):>4s}  "
          f"n_OH={info[3]} n_keto={info[4]}")

print(f"\nBile acid encoding: {BA_DIM} dims")
print(f"Features: {BA_FEATURE_NAMES}")

In [ ]:
%%time
# --- Build feature matrices for all combinations ---
# Each sample = (Enzyme, Amine, Hydroxyl) triple
# Features = enzyme_emb + amine_emb + bile_acid_emb

feature_matrices = {}
label_schemes = ['active_approach2', 'active_majority']

for enz_name, enz_dict in enzyme_repr.items():
    for ami_name, (ami_dict, ami_dim) in amine_reprs.items():
        combo_key = f"{enz_name}__{ami_name}"
        
        X_list, enzymes_list, amines_list, hydroxyls_list = [], [], [], []
        labels = {ls: [] for ls in label_schemes}
        
        for _, row in df_activity.iterrows():
            enzyme = row['Enzyme']
            amine = row['Amine']
            hydroxyl = row['Hydroxyl']
            
            if enzyme not in enz_dict or amine not in ami_dict:
                continue
            
            enz_emb = enz_dict[enzyme]
            amine_emb = ami_dict[amine]
            ba_emb = bile_acid_encodings[hydroxyl]
            
            X_list.append(np.concatenate([enz_emb, amine_emb, ba_emb]))
            enzymes_list.append(enzyme)
            amines_list.append(amine)
            hydroxyls_list.append(hydroxyl)
            for ls in label_schemes:
                labels[ls].append(int(row[ls]))
        
        X = np.array(X_list, dtype=np.float32)
        enz_dim = list(enz_dict.values())[0].shape[0]
        
        feature_matrices[combo_key] = {
            'X': X,
            'labels': {ls: np.array(v, dtype=np.int32) for ls, v in labels.items()},
            'enzymes': enzymes_list,
            'amines': amines_list,
            'hydroxyls': hydroxyls_list,
            'enz_dim': enz_dim,
            'amine_dim': ami_dim,
            'ba_dim': BA_DIM,
        }
        
        y_a2 = np.array(labels['active_approach2'])
        y_maj = np.array(labels['active_majority'])
        print(f"{combo_key}: X={X.shape}, approach2={y_a2.mean():.1%}, majority={y_maj.mean():.1%}")

print(f"\nTotal combinations: {len(feature_matrices)}")

## Section 5: Train Product-Level Models

In [ ]:
%%time
# --- 120 experiments: 2 enzyme × 3 amine × 2 labels × 10 seeds ---

all_results = []
all_importances = []  # store feature importances per experiment

combo_keys = list(feature_matrices.keys())
total_exps = len(combo_keys) * len(label_schemes) * len(SEEDS)
exp_count = 0

for combo_key in combo_keys:
    fm = feature_matrices[combo_key]
    X = fm['X']
    enzymes = fm['enzymes']
    enz_name, ami_name = combo_key.split('__')
    enz_dim = fm['enz_dim']
    amine_dim = fm['amine_dim']
    ba_dim = fm['ba_dim']
    
    for label_scheme in label_schemes:
        y = fm['labels'][label_scheme]
        
        for i, seed in enumerate(SEEDS):
            split = enzyme_holdout_split_seed(X, y, enzymes, seed=seed)
            metrics, model = train_xgb_product(split)
            
            metrics['combo'] = combo_key
            metrics['enzyme_repr'] = enz_name
            metrics['amine_repr'] = ami_name
            metrics['label_scheme'] = label_scheme
            metrics['split_idx'] = i
            metrics['split_seed'] = seed
            metrics['enz_dim'] = enz_dim
            metrics['amine_dim'] = amine_dim
            metrics['ba_dim'] = ba_dim
            metrics['total_dims'] = X.shape[1]
            all_results.append(metrics)
            
            # Store feature importances by block
            imp = model.feature_importances_
            enz_imp = imp[:enz_dim].sum()
            ami_imp = imp[enz_dim:enz_dim+amine_dim].sum()
            ba_imp = imp[enz_dim+amine_dim:].sum()
            total_imp = imp.sum()
            
            all_importances.append({
                'combo': combo_key,
                'enzyme_repr': enz_name,
                'amine_repr': ami_name,
                'label_scheme': label_scheme,
                'split_idx': i,
                'enzyme_frac': enz_imp / total_imp if total_imp > 0 else 0,
                'amine_frac': ami_imp / total_imp if total_imp > 0 else 0,
                'ba_frac': ba_imp / total_imp if total_imp > 0 else 0,
                'importances': imp,  # full vector for per-feature analysis
            })
            
            exp_count += 1
        
        # Progress report per combo × label
        combo_results = [r for r in all_results 
                        if r['combo'] == combo_key and r['label_scheme'] == label_scheme]
        roc_mean = np.mean([r['roc_auc'] for r in combo_results])
        pr_mean = np.mean([r['pr_auc'] for r in combo_results])
        print(f"[{exp_count}/{total_exps}] {combo_key} | {label_scheme}: "
              f"ROC={roc_mean:.3f}, PR={pr_mean:.3f}")

df_results = pd.DataFrame(all_results)
df_imp = pd.DataFrame([{k: v for k, v in d.items() if k != 'importances'} for d in all_importances])

print(f"\nTotal experiments: {len(df_results)}")
print("Done!")

In [ ]:
# --- Save full results ---
df_results.to_csv(PRODUCT_DIR / 'product_model_results.csv', index=False)
print(f"Saved: {PRODUCT_DIR / 'product_model_results.csv'} ({len(df_results)} rows)")

## Section 6: Three-Block Feature Importance

In [ ]:
# --- Three-block importance summary ---
print("Feature Importance by Block (mean ± std across 10 seeds)")
print("=" * 100)

imp_summary_rows = []
for combo_key in combo_keys:
    enz_name, ami_name = combo_key.split('__')
    for label_scheme in label_schemes:
        mask = (df_imp['combo'] == combo_key) & (df_imp['label_scheme'] == label_scheme)
        sub = df_imp[mask]
        row = {
            'enzyme_repr': enz_name,
            'amine_repr': ami_name,
            'label_scheme': label_scheme,
            'enzyme_pct_mean': sub['enzyme_frac'].mean() * 100,
            'enzyme_pct_std': sub['enzyme_frac'].std() * 100,
            'amine_pct_mean': sub['amine_frac'].mean() * 100,
            'amine_pct_std': sub['amine_frac'].std() * 100,
            'ba_pct_mean': sub['ba_frac'].mean() * 100,
            'ba_pct_std': sub['ba_frac'].std() * 100,
        }
        imp_summary_rows.append(row)
        print(f"{enz_name:>15} + {ami_name:<18} [{label_scheme}]: "
              f"Enzyme={row['enzyme_pct_mean']:.1f}±{row['enzyme_pct_std']:.1f}%  "
              f"Amine={row['amine_pct_mean']:.1f}±{row['amine_pct_std']:.1f}%  "
              f"BileAcid={row['ba_pct_mean']:.1f}±{row['ba_pct_std']:.1f}%")

df_imp_summary = pd.DataFrame(imp_summary_rows)
df_imp_summary.to_csv(PRODUCT_DIR / 'feature_importance_summary.csv', index=False)
print(f"\nSaved: {PRODUCT_DIR / 'feature_importance_summary.csv'}")

In [ ]:
# --- Figure: Three-block feature importance (stacked bars) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

colors = {'Enzyme': '#3498db', 'Amine': '#e74c3c', 'Bile Acid': '#2ecc71'}

for ax, label_scheme in zip(axes, label_schemes):
    sub = df_imp_summary[df_imp_summary['label_scheme'] == label_scheme]
    labels_x = [f"{r['enzyme_repr']}\n+ {r['amine_repr']}" for _, r in sub.iterrows()]
    x = np.arange(len(labels_x))
    
    enz_vals = sub['enzyme_pct_mean'].values
    ami_vals = sub['amine_pct_mean'].values
    ba_vals = sub['ba_pct_mean'].values
    
    ax.bar(x, enz_vals, color=colors['Enzyme'], label='Enzyme', edgecolor='black', linewidth=0.3)
    ax.bar(x, ami_vals, bottom=enz_vals, color=colors['Amine'], label='Amine', edgecolor='black', linewidth=0.3)
    ax.bar(x, ba_vals, bottom=enz_vals + ami_vals, color=colors['Bile Acid'], label='Bile Acid', edgecolor='black', linewidth=0.3)
    
    # Add percentage labels
    for i in range(len(x)):
        if enz_vals[i] > 5:
            ax.text(i, enz_vals[i]/2, f"{enz_vals[i]:.0f}%", ha='center', va='center', fontsize=9, fontweight='bold')
        if ami_vals[i] > 3:
            ax.text(i, enz_vals[i] + ami_vals[i]/2, f"{ami_vals[i]:.1f}%", ha='center', va='center', fontsize=8)
        if ba_vals[i] > 2:
            ax.text(i, enz_vals[i] + ami_vals[i] + ba_vals[i]/2, f"{ba_vals[i]:.1f}%", ha='center', va='center', fontsize=8)
    
    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, fontsize=9)
    ax.set_ylabel('Feature Importance (%)', fontsize=11)
    ax.set_title(f'Three-Block Importance\n({label_scheme})', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Feature Importance: Enzyme vs Amine vs Bile Acid Core', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'feature_importance_three_blocks.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'feature_importance_three_blocks.png'}")

In [ ]:
# --- Top features per block (for best config) ---
# Use the best combo (by PR-AUC, approach2 label) to show per-feature detail
best_combo = df_results[df_results['label_scheme'] == 'active_approach2'].groupby('combo')['pr_auc'].mean().idxmax()
best_label = 'active_approach2'
print(f"Best combo for per-feature analysis: {best_combo}")

# Collect importances across seeds for this combo
best_imps = [d['importances'] for d in all_importances 
             if d['combo'] == best_combo and d['label_scheme'] == best_label]
imp_matrix = np.array(best_imps)  # (10, total_dims)
imp_mean = imp_matrix.mean(axis=0)
imp_std = imp_matrix.std(axis=0)

fm = feature_matrices[best_combo]
enz_dim = fm['enz_dim']
amine_dim = fm['amine_dim']
ba_dim = fm['ba_dim']

# Create feature name arrays
enz_features = [f'enz_{i}' for i in range(enz_dim)]
amine_features = [f'ami_{i}' for i in range(amine_dim)]
ba_features = BA_FEATURE_NAMES

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Top 10 enzyme features
enz_imp = imp_mean[:enz_dim]
enz_imp_std = imp_std[:enz_dim]
top_enz = np.argsort(enz_imp)[-10:][::-1]
axes[0].barh(range(10), enz_imp[top_enz], xerr=enz_imp_std[top_enz],
             color='#3498db', alpha=0.8, capsize=3, edgecolor='black', linewidth=0.3)
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([enz_features[i] for i in top_enz], fontsize=9)
axes[0].set_xlabel('Importance', fontsize=10)
axes[0].set_title(f'Top 10 Enzyme Features\n(of {enz_dim})', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()

# Top 10 amine features
ami_imp = imp_mean[enz_dim:enz_dim+amine_dim]
ami_imp_std = imp_std[enz_dim:enz_dim+amine_dim]
top_ami = np.argsort(ami_imp)[-min(10, amine_dim):][::-1]
axes[1].barh(range(len(top_ami)), ami_imp[top_ami], xerr=ami_imp_std[top_ami],
             color='#e74c3c', alpha=0.8, capsize=3, edgecolor='black', linewidth=0.3)
axes[1].set_yticks(range(len(top_ami)))
axes[1].set_yticklabels([amine_features[i] for i in top_ami], fontsize=9)
axes[1].set_xlabel('Importance', fontsize=10)
axes[1].set_title(f'Top Amine Features\n(of {amine_dim})', fontsize=12, fontweight='bold')
axes[1].invert_yaxis()

# All bile acid features (only 12)
ba_imp = imp_mean[enz_dim+amine_dim:]
ba_imp_std = imp_std[enz_dim+amine_dim:]
sort_idx = np.argsort(ba_imp)[::-1]
axes[2].barh(range(ba_dim), ba_imp[sort_idx], xerr=ba_imp_std[sort_idx],
             color='#2ecc71', alpha=0.8, capsize=3, edgecolor='black', linewidth=0.3)
axes[2].set_yticks(range(ba_dim))
axes[2].set_yticklabels([ba_features[i] for i in sort_idx], fontsize=9)
axes[2].set_xlabel('Importance', fontsize=10)
axes[2].set_title(f'All Bile Acid Features\n({ba_dim} total)', fontsize=12, fontweight='bold')
axes[2].invert_yaxis()

plt.suptitle(f'Per-Feature Importance ({best_combo})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'top_features_per_block.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'top_features_per_block.png'}")

## Section 7: Representation Comparisons

In [ ]:
# --- Summary table ---
summary_rows = []
for combo_key in combo_keys:
    enz_name, ami_name = combo_key.split('__')
    for label_scheme in label_schemes:
        mask = (df_results['combo'] == combo_key) & (df_results['label_scheme'] == label_scheme)
        sub = df_results[mask]
        row = {
            'combo': combo_key,
            'enzyme_repr': enz_name,
            'amine_repr': ami_name,
            'label_scheme': label_scheme,
            'total_dims': sub['total_dims'].iloc[0],
            'roc_auc_mean': sub['roc_auc'].mean(),
            'roc_auc_std': sub['roc_auc'].std(),
            'pr_auc_mean': sub['pr_auc'].mean(),
            'pr_auc_std': sub['pr_auc'].std(),
            'f1_mean': sub['f1'].mean(),
            'f1_std': sub['f1'].std(),
            'logloss_gap_mean': sub['logloss_gap'].mean(),
            'logloss_gap_std': sub['logloss_gap'].std(),
        }
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

for label_scheme in label_schemes:
    print(f"\n{'='*110}")
    print(f"Results — {label_scheme} (ranked by PR-AUC):")
    print(f"{'='*110}")
    sub = df_summary[df_summary['label_scheme'] == label_scheme].sort_values('pr_auc_mean', ascending=False)
    print(f"{'Enzyme':<18} {'Amine':<18} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'F1':>12} {'LL Gap':>12}")
    print("-" * 100)
    for _, r in sub.iterrows():
        print(f"{r['enzyme_repr']:<18} {r['amine_repr']:<18} {r['total_dims']:>5.0f} "
              f"{r['roc_auc_mean']:.3f}±{r['roc_auc_std']:.3f} "
              f"{r['pr_auc_mean']:.3f}±{r['pr_auc_std']:.3f} "
              f"{r['f1_mean']:.3f}±{r['f1_std']:.3f} "
              f"{r['logloss_gap_mean']:>+.3f}±{r['logloss_gap_std']:.3f}")

In [ ]:
# --- Figure: Enzyme representation comparison ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

enz_colors = {'noncons_max': '#e74c3c', 'full_protein': '#3498db'}
enz_names = list(enzyme_repr.keys())
ami_names = list(amine_reprs.keys())

for ax, (metric, title) in zip(axes, 
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1')]):
    
    n_ami = len(ami_names)
    n_enz = len(enz_names)
    width = 0.8 / n_enz
    x = np.arange(n_ami)
    
    for j, enz_name in enumerate(enz_names):
        means, stds = [], []
        for ami_name in ami_names:
            combo = f"{enz_name}__{ami_name}"
            row = df_summary[(df_summary['combo'] == combo) & (df_summary['label_scheme'] == 'active_approach2')]
            if len(row) > 0:
                means.append(row[f'{metric}_mean'].values[0])
                stds.append(row[f'{metric}_std'].values[0])
            else:
                means.append(0); stds.append(0)
        
        offset = (j - n_enz/2 + 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=enz_name,
               color=enz_colors.get(enz_name, 'gray'), alpha=0.8, capsize=3,
               edgecolor='black', linewidth=0.3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(ami_names, fontsize=10)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Enzyme Representation Comparison (active_approach2)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'enzyme_representation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'enzyme_representation_comparison.png'}")

In [ ]:
# --- Figure: Amine representation comparison ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ami_colors = {'physchem_onehot': '#3498db', 'molt5_base': '#2ecc71', 'hybrid': '#9b59b6'}

for ax, (metric, title) in zip(axes,
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1')]):
    
    n_enz = len(enz_names)
    n_ami = len(ami_names)
    width = 0.8 / n_ami
    x = np.arange(n_enz)
    
    for j, ami_name in enumerate(ami_names):
        means, stds = [], []
        for enz_name in enz_names:
            combo = f"{enz_name}__{ami_name}"
            row = df_summary[(df_summary['combo'] == combo) & (df_summary['label_scheme'] == 'active_approach2')]
            if len(row) > 0:
                means.append(row[f'{metric}_mean'].values[0])
                stds.append(row[f'{metric}_std'].values[0])
            else:
                means.append(0); stds.append(0)
        
        offset = (j - n_ami/2 + 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=ami_name,
               color=ami_colors.get(ami_name, 'gray'), alpha=0.8, capsize=3,
               edgecolor='black', linewidth=0.3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(enz_names, fontsize=10)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Amine Representation Comparison (active_approach2)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'amine_representation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'amine_representation_comparison.png'}")

In [ ]:
# --- Figure: Log loss gap comparison (overfitting check) ---
fig, ax = plt.subplots(figsize=(12, 6))

sub = df_summary[df_summary['label_scheme'] == 'active_approach2'].sort_values('logloss_gap_mean')
labels_x = [f"{r['enzyme_repr']}\n+ {r['amine_repr']}" for _, r in sub.iterrows()]
x = np.arange(len(labels_x))

colors_gap = ['#e74c3c' if v > 0 else '#3498db' for v in sub['logloss_gap_mean']]
ax.bar(x, sub['logloss_gap_mean'], yerr=sub['logloss_gap_std'],
       color=colors_gap, alpha=0.8, capsize=3, edgecolor='black', linewidth=0.3)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=10)
ax.set_ylabel('Log Loss Gap (val - train)', fontsize=11)
ax.set_title('Overfitting Check: Log Loss Gap (active_approach2)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'amine_representation_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'amine_representation_gap.png'}")

## Section 8: Label Comparison & Summary

In [ ]:
# --- Figure: Label scheme comparison ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

label_colors = {'active_approach2': '#3498db', 'active_majority': '#e74c3c'}

for ax, (metric, title) in zip(axes,
    [('roc_auc', 'ROC-AUC'), ('pr_auc', 'PR-AUC'), ('f1', 'F1')]):
    
    n_ls = len(label_schemes)
    width = 0.8 / n_ls
    x = np.arange(len(combo_keys))
    
    for j, label_scheme in enumerate(label_schemes):
        means, stds = [], []
        for combo_key in combo_keys:
            row = df_summary[(df_summary['combo'] == combo_key) & (df_summary['label_scheme'] == label_scheme)]
            if len(row) > 0:
                means.append(row[f'{metric}_mean'].values[0])
                stds.append(row[f'{metric}_std'].values[0])
            else:
                means.append(0); stds.append(0)
        
        offset = (j - n_ls/2 + 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=label_scheme,
               color=label_colors[label_scheme], alpha=0.8, capsize=3,
               edgecolor='black', linewidth=0.3)
    
    xtick_labels = [k.replace('__', '\n+ ') for k in combo_keys]
    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, fontsize=8)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Label Scheme Comparison: approach2 vs majority vote', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(PRODUCT_DIR / 'label_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {PRODUCT_DIR / 'label_comparison.png'}")

In [ ]:
# --- Final summary ---
df_summary.to_csv(PRODUCT_DIR / 'product_model_summary.csv', index=False)
print(f"Saved: {PRODUCT_DIR / 'product_model_summary.csv'} ({len(df_summary)} rows)")

print("\n" + "=" * 110)
print("BSH PRODUCT-LEVEL MODEL — FINAL SUMMARY")
print("=" * 110)

for label_scheme in label_schemes:
    print(f"\n--- {label_scheme} (ranked by PR-AUC) ---")
    sub = df_summary[df_summary['label_scheme'] == label_scheme].sort_values('pr_auc_mean', ascending=False)
    print(f"{'Rank':>4} {'Enzyme':<18} {'Amine':<18} {'Dims':>5} {'ROC-AUC':>14} {'PR-AUC':>14} {'LL Gap':>12}")
    print("-" * 95)
    for rank, (_, r) in enumerate(sub.iterrows(), 1):
        marker = ' ***' if rank == 1 else ''
        print(f"{rank:>4} {r['enzyme_repr']:<18} {r['amine_repr']:<18} {r['total_dims']:>5.0f} "
              f"{r['roc_auc_mean']:.3f}±{r['roc_auc_std']:.3f} "
              f"{r['pr_auc_mean']:.3f}±{r['pr_auc_std']:.3f} "
              f"{r['logloss_gap_mean']:>+.3f}±{r['logloss_gap_std']:.3f}{marker}")

# Best overall
best = df_summary.sort_values('pr_auc_mean', ascending=False).iloc[0]
print(f"\n{'='*110}")
print(f"BEST OVERALL: {best['enzyme_repr']} + {best['amine_repr']} ({best['label_scheme']})")
print(f"  ROC-AUC: {best['roc_auc_mean']:.3f} ± {best['roc_auc_std']:.3f}")
print(f"  PR-AUC:  {best['pr_auc_mean']:.3f} ± {best['pr_auc_std']:.3f}")
print(f"  F1:      {best['f1_mean']:.3f} ± {best['f1_std']:.3f}")
print(f"  LL Gap:  {best['logloss_gap_mean']:+.3f}")
print(f"  Dims:    {best['total_dims']:.0f} (enzyme + amine + bile acid)")
print(f"{'='*110}")

# Feature importance for best config
best_combo_name = f"{best['enzyme_repr']}__{best['amine_repr']}"
best_imp = df_imp_summary[
    (df_imp_summary['enzyme_repr'] == best['enzyme_repr']) & 
    (df_imp_summary['amine_repr'] == best['amine_repr']) &
    (df_imp_summary['label_scheme'] == best['label_scheme'])
]
if len(best_imp) > 0:
    bi = best_imp.iloc[0]
    print(f"\nFeature importance (best config):")
    print(f"  Enzyme:    {bi['enzyme_pct_mean']:.1f} ± {bi['enzyme_pct_std']:.1f}%")
    print(f"  Amine:     {bi['amine_pct_mean']:.1f} ± {bi['amine_pct_std']:.1f}%")
    print(f"  Bile Acid: {bi['ba_pct_mean']:.1f} ± {bi['ba_pct_std']:.1f}%")

print(f"\nAll outputs saved to: {PRODUCT_DIR}")
print("\nExpected outputs:")
print("  - product_model_results.csv (120 experiments)")
print("  - product_model_summary.csv (aggregated)")
print("  - replicate_label_comparison.csv")
print("  - feature_importance_summary.csv")
print("  - feature_importance_three_blocks.png")
print("  - top_features_per_block.png")
print("  - enzyme_representation_comparison.png")
print("  - amine_representation_comparison.png")
print("  - amine_representation_gap.png")
print("  - label_comparison.png")